# Content Based Recommender Systems

In [23]:
from math import *

def square_rooted(x):
   return round(sqrt(sum([a*a for a in x])), 3)

def cosine_similarity(x,y):
   numerator = sum(a*b for a,b in zip(x,y))
   denominator = square_rooted(x)*square_rooted(y)
   return round(numerator/float(denominator),3)

In [24]:
print(cosine_similarity([0.5,0.5], [0.1, 0.1]))

1.003


STEPS:
1. Convert the unstructured data to a dtructured formart(document term frequency matrix)
2. Compute rhe cosine similarity between user search item and rest of the products


- Document Term Frequency Matrix

We want to base on our past movie data to recommend similar movies based on similar plot, genre, director, etc

In [25]:
# importing necessary libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer


pd.set_option('display.max_columns', 100)
df = pd.read_csv('https://query.data.world/s/uikepcpffyo2nhig52xxeevdialfl7')

In [26]:
df.head()

,Unnamed: 0,Title,Year,Rated,Released,Runtime,Genre,Director,Writer,Actors,Plot,Language,Country,Awards,Poster,Ratings.Source,Ratings.Value,Metascore,imdbRating,imdbVotes,imdbID,Type,tomatoMeter,tomatoImage,tomatoRating,tomatoReviews,tomatoFresh,tomatoRotten,tomatoConsensus,tomatoUserMeter,tomatoUserRating,tomatoUserReviews,tomatoURL,DVD,BoxOffice,Production,Website,Response
0,1,The Shawshank Redemption,1994,R,14 Oct 1994,142 min,"Crime, Drama",Frank Darabont,"Stephen King (short story ""Rita Hayworth and S...","Tim Robbins, Morgan Freeman, Bob Gunton, Willi...",Two imprisoned men bond over a number of years...,English,USA,Nominated for 7 Oscars. Another 19 wins & 30 n...,https://images-na.ssl-images-amazon.com/images...,Internet Movie Database,9.3/10,80.0,9.3,"1,825,626",tt0111161,movie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.rottentomatoes.com/m/shawshank_rede...,27 Jan 1998,NaN,Columbia Pictures,NaN,True
1,2,The Godfather,1972,R,24 Mar 1972,175 min,"Crime, Drama",Francis Ford Coppola,"Mario Puzo (screenplay), Francis Ford Coppola ...","Marlon Brando, Al Pacino, James Caan, Richard ...",The aging patriarch of an organized crime dyna...,"English, Italian, Latin",USA,Won 3 Oscars. Another 23 wins & 27 nominations.,https://images-na.ssl-images-amazon.com/images...,Internet Movie Database,9.2/10,100.0,9.2,"1,243,444",tt0068646,movie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.rottentomatoes.com/m/godfather/,09 Oct 2001,NaN,Paramount Pictures,http://www.thegodfather.com,True
2,3,The Godfather: Part II,1974,R,20 Dec 1974,202 min,"Crime, Drama",Francis Ford Coppola,"Francis Ford Coppola (screenplay), Mario Puzo ...","Al Pacino, Robert Duvall, Diane Keaton, Robert...",The early life and career of Vito Corleone in ...,"English, Italian, Spanish, Latin, Sicilian",USA,Won 6 Oscars. Another 10 wins & 20 nominations.,https://images-na.ssl-images-amazon.com/images...,Internet Movie Database,9.0/10,85.0,9.0,"856,870",tt0071562,movie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.rottentomatoes.com/m/godfather_part...,24 May 2005,NaN,Paramount Pictures,http://www.thegodfather.com/,True
3,4,The Dark Knight,2008,PG-13,18 Jul 2008,152 min,"Action, Crime, Drama",Christopher Nolan,"Jonathan Nolan (screenplay), Christopher Nolan...","Christian Bale, Heath Ledger, Aaron Eckhart, M...",When the menace known as the Joker emerges fro...,"English, Mandarin","USA, UK",Won 2 Oscars. Another 151 wins & 153 nominations.,https://images-na.ssl-images-amazon.com/images...,Internet Movie Database,9.0/10,82.0,9.0,"1,802,351",tt0468569,movie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.rottentomatoes.com/m/the_dark_knight/,09 Dec 2008,"$533,316,061",Warner Bros. Pictures/Legendary,http://thedarkknight.warnerbros.com/,True
4,5,12 Angry Men,1957,APPROVED,01 Apr 1957,96 min,"Crime, Drama",Sidney Lumet,"Reginald Rose (story), Reginald Rose (screenplay)","Martin Balsam, John Fiedler, Lee J. Cobb, E.G....",A jury holdout attempts to prevent a miscarria...,English,USA,Nominated for 3 Oscars. Another 16 wins & 8 no...,https://images-na.ssl-images-amazon.com/images...,Internet Movie Database,8.9/10,96.0,8.9,"494,215",tt0050083,movie,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,http://www.rottentomatoes.com/m/1000013-12_ang...,06 Mar 2001,NaN,Criterion Collection,http://www.criterion.com/films/27871-12-angry-men,True


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 38 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Unnamed: 0         250 non-null    int64  
 1   Title              250 non-null    object 
 2   Year               250 non-null    int64  
 3   Rated              250 non-null    object 
 4   Released           248 non-null    object 
 5   Runtime            250 non-null    object 
 6   Genre              250 non-null    object 
 7   Director           250 non-null    object 
 8   Writer             249 non-null    object 
 9   Actors             250 non-null    object 
 10  Plot               250 non-null    object 
 11  Language           250 non-null    object 
 12  Country            250 non-null    object 
 13  Awards             245 non-null    object 
 14  Poster             250 non-null    object 
 15  Ratings.Source     250 non-null    object 
 16  Ratings.Value      250 non

We will base our recommendation on "Title", "Genre", "writer", "Actors", and "Plot"

In [28]:
df = df[["Title", "Genre", "Actors",'Director', "Plot"]]
df.head()

,Title,Genre,Actors,Director,Plot
0,The Shawshank Redemption,"Crime, Drama","Tim Robbins, Morgan Freeman, Bob Gunton, Willi...",Frank Darabont,Two imprisoned men bond over a number of years...
1,The Godfather,"Crime, Drama","Marlon Brando, Al Pacino, James Caan, Richard ...",Francis Ford Coppola,The aging patriarch of an organized crime dyna...
2,The Godfather: Part II,"Crime, Drama","Al Pacino, Robert Duvall, Diane Keaton, Robert...",Francis Ford Coppola,The early life and career of Vito Corleone in ...
3,The Dark Knight,"Action, Crime, Drama","Christian Bale, Heath Ledger, Aaron Eckhart, M...",Christopher Nolan,When the menace known as the Joker emerges fro...
4,12 Angry Men,"Crime, Drama","Martin Balsam, John Fiedler, Lee J. Cobb, E.G....",Sidney Lumet,A jury holdout attempts to prevent a miscarria...


In [29]:
# discarding the commas between the actors' full names and getting only the first three names
df['Actors']= df['Actors'].map(lambda x: x.lower().replace(' ', '').split(','))

In [30]:
df['Genre'] = df['Genre'].map(lambda x: x.lower().split(','))

In [31]:
df['Genre']

0                 [crime,  drama]
1                 [crime,  drama]
2                 [crime,  drama]
3        [action,  crime,  drama]
4                 [crime,  drama]
                  ...            
245           [drama,  film-noir]
246                       [drama]
247    [comedy,  drama,  romance]
248           [biography,  drama]
249                       [drama]
Name: Genre, Length: 250, dtype: object

In [32]:
df['Director'] = df['Director'].map(lambda x: x.lower().split(' '))

In [33]:
df['Director'] = df["Director"].apply("".join)

In [34]:
df['Director']

0                  frankdarabont
1             francisfordcoppola
2             francisfordcoppola
3               christophernolan
4                    sidneylumet
                 ...            
245                  billywilder
246          destindanielcretton
247                  howardhawks
248                   davidlynch
249    dannyboyle,loveleentandan
Name: Director, Length: 250, dtype: object

In [35]:
df['Actors']

0      [timrobbins, morganfreeman, bobgunton, william...
1      [marlonbrando, alpacino, jamescaan, richards.c...
2      [alpacino, robertduvall, dianekeaton, robertde...
3      [christianbale, heathledger, aaroneckhart, mic...
4      [martinbalsam, johnfiedler, leej.cobb, e.g.mar...
                             ...                        
245    [raymilland, janewyman, phillipterry, howardda...
246    [brielarson, johngallagherjr., stephaniebeatri...
247    [carygrant, rosalindrussell, ralphbellamy, gen...
248    [sissyspacek, janegallowayheitz, josepha.carpe...
249     [devpatel, saurabhshukla, anilkapoor, rajzutshi]
Name: Actors, Length: 250, dtype: object

In [36]:
import rake_nltk 
from rake_nltk import Rake # help extract keywords that are put together
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to C:\Users\Latifa
[nltk_data]     Riziki\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to C:\Users\Latifa
[nltk_data]     Riziki\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [37]:
df['Key_words'] = ""

for index, row in df.iterrows():

   plot = row['Plot']

   # instantiating Rake, by deafault it uses english stopwords
   r= Rake()

   #extracting the words by passing thebtext
   r.extract_keywords_from_text(plot)

   # getting the dictionary with key words and their scores
   key_words_dict_scores = r.get_word_degrees()

   # assigning the key words to the new columnb
   row['key_words'] = list(key_words_dict_scores.keys())

df.drop(columns = ['Plot'], inplace = True)

In [38]:
key_words_dict_scores

defaultdict(<function rake_nltk.rake.Rake._build_word_co_occurance_graph.<locals>.<lambda>()>,
            {'mumbai': 3,
             'teen': 3,
             'reflects': 3,
             'upbringing': 1,
             'slums': 1,
             'accused': 1,
             'cheating': 1,
             'indian': 2,
             'version': 2,
             'wants': 1,
             'millionaire': 2,
             '?"': 2})

In [39]:
df.set_index('Title', inplace= True)

In [40]:
print(type(df.iloc[0][df.columns[0]]))


<class 'list'>


In [41]:
df['bag_of_words'] = ' '
columns = df.columns
for index, row in df.iterrows():
   words = ''
   for col in columns:
      if col != 'Director':
         words += ' '.join(row[col])+ ' '
      else:
         words = words + row[col] + ' '
   df.at[index, 'bag_of_words'] = words.strip()

df.drop(columns = [ col for col in df.columns if col != 'bag_of_words'], inplace = True)


In [42]:
count = CountVectorizer()
count_matrix = count.fit_transform(df['bag_of_words'])

In [43]:
count_matrix

<250x980 sparse matrix of type '<class 'numpy.int64'>'
	with 1974 stored elements in Compressed Sparse Row format>

In [44]:
c = count_matrix.todense()

In [45]:
c

matrix([[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]], dtype=int64)

In [46]:
print(count_matrix[0,:])

  (0, 174)	1
  (0, 234)	1
  (0, 902)	1
  (0, 669)	1
  (0, 89)	1
  (0, 968)	1
  (0, 297)	1


In [47]:
# generating the cosine similarity matrix
cosine_sim = cosine_similarity(count_matrix, count_matrix)
cosine_sim

array([[1.        , 0.26726124, 0.28571429, ..., 0.13363062, 0.13363062,
        0.14285714],
       [0.26726124, 1.        , 0.53452248, ..., 0.125     , 0.125     ,
        0.13363062],
       [0.28571429, 0.53452248, 1.        , ..., 0.13363062, 0.13363062,
        0.14285714],
       ...,
       [0.13363062, 0.125     , 0.13363062, ..., 1.        , 0.125     ,
        0.13363062],
       [0.13363062, 0.125     , 0.13363062, ..., 0.125     , 1.        ,
        0.13363062],
       [0.14285714, 0.13363062, 0.14285714, ..., 0.13363062, 0.13363062,
        1.        ]])

In [48]:
# creating a Series for the movie titles so they are associated to an orderef numerical list
indices = pd.Series(df.index)
indices[:20]

0                              The Shawshank Redemption
1                                         The Godfather
2                                The Godfather: Part II
3                                       The Dark Knight
4                                          12 Angry Men
5                                      Schindler's List
6         The Lord of the Rings: The Return of the King
7                                          Pulp Fiction
8                                            Fight Club
9     The Lord of the Rings: The Fellowship of the Ring
10                                         Forrest Gump
11       Star Wars: Episode V - The Empire Strikes Back
12                                            Inception
13                The Lord of the Rings: The Two Towers
14                      One Flew Over the Cuckoo's Nest
15                                           Goodfellas
16                                           The Matrix
17                   Star Wars: Episode IV - A N

In [51]:
# function that takes in movie title as input and returns the top 10 recommended movies
def recommendations(title, cosine_sim = cosine_sim):
   recommended_movies = []

   #getting the index of the movie that matches the title
   idx= indices[indices == title].index[0]
   #creating a series with the similarity scores in descending order
   score_series = pd.Series(cosine_sim[idx]).sort_values(ascending = False)

   # getting the indexes of the 10 most similar movies
   top_10_indices = list(score_series.iloc[1:11].index)
   print(top_10_indices)

   # populating the list with the titles of the best 10 matching movies
   for i in top_10_indices:
      recommended_movies.append(list(df.index)[i])
   
   return recommended_movies

In [53]:
recommendations('Fargo')

[132, 125, 226, 61, 34, 22, 100, 20, 239, 123]


['No Country for Old Men',
 'The Big Lebowski',
 'Rope',
 'Reservoir Dogs',
 'The Departed',
 'Léon: The Professional',
 'On the Waterfront',
 'The Silence of the Lambs',
 'The Manchurian Candidate',
 'Cool Hand Luke']